# Dataset

Here, we will deal with the validation of the dataset and making sure that examples are proper before we start training.

In [ ]:
import os
import json
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

In [ ]:



# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = "./HD-EPIC"
DATASET_DIR = f"{ROOT}/Dataset"
FEATURES_DIR = f"{DATASET_DIR}/Videos"
MANIFEST_PATH = f"{DATASET_DIR}/split_manifest.json"

SEQUENCE_LENGTH = 11 # 11 clips give 10 seconds worth of video
FEATURE_DIM = 768
NUM_CLASSES = 304
PREDICTION_HORIZON = 30.0

# Features should be sampled approximately once per second.
EXPECTED_INTERVAL = 1.0
TIMESTAMP_TOLERANCE = 0.05

BATCH_SIZE = 32
RANDOM_SEED = 42

SHORT_VIDEO_ID = "P09-20240623-154547"

assert os.path.isdir(FEATURES_DIR), f"Missing directory: {FEATURES_DIR}"
assert os.path.isfile(MANIFEST_PATH), f"Missing split manifest: {MANIFEST_PATH}"


# ------------------------------------------------------------
# 2. Load and validate the existing split manifest
# ------------------------------------------------------------

with open(MANIFEST_PATH, "r") as f:
    split_manifest = json.load(f)

required_splits = {"train", "validation", "test"}

assert set(split_manifest.keys()) == required_splits, (
    f"Expected exactly these splits: {required_splits}"
)

split_ids = {}

for split_name in required_splits:
    split_ids[split_name] = list(split_manifest[split_name])

    assert len(split_ids[split_name]) == len(set(split_ids[split_name])), (
        f"Duplicate video IDs in {split_name} split."
    )

train_ids = set(split_ids["train"])
validation_ids = set(split_ids["validation"])
test_ids = set(split_ids["test"])

assert train_ids.isdisjoint(validation_ids), "Train/validation overlap."
assert train_ids.isdisjoint(test_ids), "Train/test overlap."
assert validation_ids.isdisjoint(test_ids), "Validation/test overlap."

print("Existing split manifest loaded.")
print(f"Training videos:   {len(train_ids)}")
print(f"Validation videos: {len(validation_ids)}")
print(f"Test videos:       {len(test_ids)}")


# ------------------------------------------------------------
# 3. Find saved video datasets
# ------------------------------------------------------------

available_files = sorted(
    filename
    for filename in os.listdir(FEATURES_DIR)
    if filename.endswith(".npz")
)

available_ids = {
    os.path.splitext(filename)[0]
    for filename in available_files
}

manifest_ids = train_ids | validation_ids | test_ids

missing_ids = manifest_ids - available_ids
extra_ids = available_ids - manifest_ids

assert not missing_ids, f"Videos in manifest but missing files: {sorted(missing_ids)}"

if extra_ids:
    print(f"Warning: {len(extra_ids)} saved videos are absent from the manifest.")
    print("These videos will not be used.")

print(f"\nSaved video files: {len(available_ids)}")
print(f"Videos in manifest: {len(manifest_ids)}")


# ------------------------------------------------------------
# 4. Load and validate every video dataset
# ------------------------------------------------------------

video_data = {}

total_observations = 0
total_positive_pairs = 0

for video_id in sorted(manifest_ids):
    file_path = os.path.join(FEATURES_DIR, f"{video_id}.npz")

    with np.load(file_path) as data:
        required_keys = {
            "features",
            "observation_times",
            "relevance",
            "time_to_relevance",
            "time_mask",
            "video_duration"
        }

        assert required_keys.issubset(data.files), (
            f"{video_id}: missing keys {required_keys - set(data.files)}"
        )

        # Copy the arrays so they remain available after the file is closed.
        features = data["features"].astype(np.float32, copy=True)
        observation_times = data["observation_times"].astype(np.float32, copy=True)
        relevance = data["relevance"].astype(np.float32, copy=True)
        time_to_relevance = data["time_to_relevance"].astype(np.float32, copy=True)
        time_mask = data["time_mask"].astype(np.float32, copy=True)
        video_duration = float(data["video_duration"])

    n = len(observation_times)

    # All arrays must agree on the number of observations.
    assert features.shape == (n, FEATURE_DIM), (
        f"{video_id}: unexpected feature shape {features.shape}"
    )

    assert relevance.shape == (n, NUM_CLASSES), (
        f"{video_id}: unexpected relevance shape {relevance.shape}"
    )

    assert time_to_relevance.shape == (n, NUM_CLASSES), (
        f"{video_id}: unexpected time-to-relevance shape {time_to_relevance.shape}"
    )

    assert time_mask.shape == (n, NUM_CLASSES), (
        f"{video_id}: unexpected time-mask shape {time_mask.shape}"
    )

    assert np.all(np.isfinite(features)), f"{video_id}: non-finite features."
    assert np.all(np.isfinite(observation_times)), f"{video_id}: non-finite timestamps."
    assert np.all(np.diff(observation_times) > 0), (
        f"{video_id}: timestamps are not strictly increasing."
    )

    assert np.all((relevance == 0) | (relevance == 1)), (
        f"{video_id}: relevance contains values other than 0 and 1."
    )

    assert np.all((time_mask == 0) | (time_mask == 1)), (
        f"{video_id}: time_mask contains values other than 0 and 1."
    )

    # Under our current label definition, time targets are valid exactly
    # where relevance is positive.
    assert np.array_equal(relevance, time_mask), (
        f"{video_id}: relevance and time_mask disagree."
    )

    positive_mask = time_mask == 1
    negative_mask = time_mask == 0

    assert np.all(time_to_relevance[positive_mask] > 0), (
        f"{video_id}: positive target has a non-positive time."
    )

    assert np.all(time_to_relevance[positive_mask] <= PREDICTION_HORIZON), (
        f"{video_id}: positive target exceeds the prediction horizon."
    )

    assert np.all(time_to_relevance[negative_mask] == 0), (
        f"{video_id}: invalid time targets should be zero."
    )

    # The labels should only describe events within the complete future horizon.
    assert np.all(
        observation_times + PREDICTION_HORIZON <= video_duration + 1e-3
    ), f"{video_id}: an observation lacks a complete future horizon."

    video_data[video_id] = {
        "features": features,
        "observation_times": observation_times,
        "relevance": relevance,
        "time_to_relevance": time_to_relevance,
        "time_mask": time_mask,
        "video_duration": video_duration
    }

    total_observations += n
    total_positive_pairs += int(relevance.sum())

print("\nAll saved video datasets passed validation.")
print(f"Videos loaded: {len(video_data)}")
print(f"Observations: {total_observations:,}")
print(f"Positive observation/class pairs: {total_positive_pairs:,}")


# ------------------------------------------------------------
# 5. Build sequences of recent feature vectors
# ------------------------------------------------------------

def find_valid_endpoints(observation_times, sequence_length=SEQUENCE_LENGTH):
    """
    Return indices that can serve as endpoints of valid input sequences.

    Each input consists of sequence_length consecutive feature vectors.
    Consecutive observation timestamps must be approximately one second apart.
    """
    valid_endpoints = []

    for end_index in range(sequence_length - 1, len(observation_times)):
        start_index = end_index - sequence_length + 1

        sequence_times = observation_times[start_index:end_index + 1]
        intervals = np.diff(sequence_times)

        # Reject sequences containing missing or irregularly spaced observations.
        if np.all(np.abs(intervals - EXPECTED_INTERVAL) <= TIMESTAMP_TOLERANCE):
            valid_endpoints.append(end_index)

    return valid_endpoints


valid_endpoints_by_video = {}
skipped_windows_by_video = {}

for video_id, arrays in video_data.items():
    n = len(arrays["observation_times"])

    valid_endpoints = find_valid_endpoints(arrays["observation_times"])

    valid_endpoints_by_video[video_id] = valid_endpoints
    skipped_windows_by_video[video_id] = max(
        0, n - SEQUENCE_LENGTH + 1
    ) - len(valid_endpoints)

print("\nSequence construction complete.")
print(f"Required input length: {SEQUENCE_LENGTH} feature vectors")
print(f"Feature dimension: {FEATURE_DIM}")
print(f"Expected timestamp interval: {EXPECTED_INTERVAL:.1f} second")


# ------------------------------------------------------------
# 6. Create PyTorch Dataset
# ------------------------------------------------------------

class EpicDataset(Dataset):
    def __init__(self, video_ids, video_data, valid_endpoints_by_video):
        self.video_data = video_data
        self.examples = []

        for video_id in sorted(video_ids):
            for end_index in valid_endpoints_by_video[video_id]:
                self.examples.append((video_id, end_index))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        video_id, end_index = self.examples[index]
        arrays = self.video_data[video_id]

        start_index = end_index - SEQUENCE_LENGTH + 1

        # Shape: [10, 768]
        features = arrays["features"][start_index:end_index + 1]

        # Targets refer to the final observation in the sequence.
        relevance = arrays["relevance"][end_index]
        time_to_relevance = arrays["time_to_relevance"][end_index]
        time_mask = arrays["time_mask"][end_index]
        observation_time = arrays["observation_times"][end_index]

        return {
            "features": torch.from_numpy(features.copy()),
            "relevance": torch.from_numpy(relevance.copy()),
            "time_to_relevance": torch.from_numpy(time_to_relevance.copy()),
            "time_mask": torch.from_numpy(time_mask.copy()),
            "observation_time": torch.tensor(observation_time, dtype=torch.float32),
            "video_id": video_id
        }


# ------------------------------------------------------------
# 7. Create train, validation, and test datasets
# ------------------------------------------------------------

train_dataset = EpicDataset(train_ids, video_data, valid_endpoints_by_video)
validation_dataset = EpicDataset(validation_ids, video_data, valid_endpoints_by_video)
test_dataset = EpicDataset(test_ids, video_data, valid_endpoints_by_video)

print("\nFinal dataset sizes:")
print(f"Training examples:   {len(train_dataset):,}")
print(f"Validation examples: {len(validation_dataset):,}")
print(f"Test examples:       {len(test_dataset):,}")
print(
    "Total examples:      "
    f"{len(train_dataset) + len(validation_dataset) + len(test_dataset):,}"
)


# ------------------------------------------------------------
# 8. Inspect the unusually short video
# ------------------------------------------------------------

if SHORT_VIDEO_ID in video_data:
    arrays = video_data[SHORT_VIDEO_ID]
    n_observations = len(arrays["observation_times"])
    n_examples = len(valid_endpoints_by_video[SHORT_VIDEO_ID])

    print(f"\nShort-video inspection: {SHORT_VIDEO_ID}")
    print(f"Video duration: {arrays['video_duration']:.2f} seconds")
    print(f"Saved observations: {n_observations}")
    print(f"Valid 10-vector sequences: {n_examples}")

    if n_observations > 0:
        print(
            "Observation-time range: "
            f"{arrays['observation_times'][0]:.2f}–"
            f"{arrays['observation_times'][-1]:.2f} seconds"
        )

    if n_examples == 0:
        print("This video contributes no examples to model training or evaluation.")
else:
    print(f"\nShort-video inspection: {SHORT_VIDEO_ID} is absent from the manifest.")


# ------------------------------------------------------------
# 9. Summarise each split
# ------------------------------------------------------------

def summarise_split(split_name, video_ids, dataset):
    observations = sum(len(video_data[v]["observation_times"]) for v in video_ids)
    positive_pairs = sum(
        int(video_data[v]["relevance"].sum()) for v in video_ids
    )
    videos_with_examples = sum(
        len(valid_endpoints_by_video[v]) > 0 for v in video_ids
    )

    print(f"\n{split_name.upper()}")
    print(f"Videos: {len(video_ids)}")
    print(f"Videos contributing examples: {videos_with_examples}")
    print(f"Saved observations: {observations:,}")
    print(f"Positive observation/class pairs: {positive_pairs:,}")
    print(f"Usable sequence examples: {len(dataset):,}")


summarise_split("train", train_ids, train_dataset)
summarise_split("validation", validation_ids, validation_dataset)
summarise_split("test", test_ids, test_dataset)


# ------------------------------------------------------------
# 10. Create DataLoaders
# ------------------------------------------------------------

generator = torch.Generator()
generator.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=generator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# 11. Verify a real batch
# ------------------------------------------------------------

if len(train_dataset) == 0:
    raise RuntimeError("No valid training examples. Check the timestamps and sequence rules.")

batch = next(iter(train_loader))

print("\nFirst training batch:")

for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")
    else:
        print(f"{key}: {len(value)} video IDs")

assert batch["features"].shape == (BATCH_SIZE, SEQUENCE_LENGTH, FEATURE_DIM) or (
    batch["features"].shape[1:] == (SEQUENCE_LENGTH, FEATURE_DIM)
)

assert batch["relevance"].shape[1:] == (NUM_CLASSES,)
assert batch["time_to_relevance"].shape[1:] == (NUM_CLASSES,)
assert batch["time_mask"].shape[1:] == (NUM_CLASSES,)

print("\nAll dataset and DataLoader checks passed.")

Existing split manifest loaded.
Training videos:   23
Validation videos: 6
Test videos:       5

Saved video files: 34
Videos in manifest: 34

All saved video datasets passed validation.
Videos loaded: 34
Observations: 43,334
Positive observation/class pairs: 125,683

Sequence construction complete.
Required input length: 11 feature vectors
Feature dimension: 768
Expected timestamp interval: 1.0 second

Final dataset sizes:
Training examples:   23,222
Validation examples: 14,617
Test examples:       5,161
Total examples:      43,000

Short-video inspection: P09-20240623-154547
Video duration: 35.40 seconds
Saved observations: 4
Valid 10-vector sequences: 0
Observation-time range: 2.00–5.00 seconds
This video contributes no examples to model training or evaluation.

TRAIN
Videos: 23
Videos contributing examples: 23
Saved observations: 23,452
Positive observation/class pairs: 74,604
Usable sequence examples: 23,222

VALIDATION
Videos: 6
Videos contributing examples: 6
Saved observations:

In [ ]:
def count_positive_classes(loader, num_classes):
    positive_counts = torch.zeros(num_classes)

    for batch in loader:
        positive_counts += batch["relevance"].sum(dim=0)

    return positive_counts


train_positive = count_positive_classes(train_loader, NUM_CLASSES)
val_positive = count_positive_classes(validation_loader, NUM_CLASSES)
test_positive = count_positive_classes(test_loader, NUM_CLASSES)

train_seen = train_positive > 0
val_seen = val_positive > 0
test_seen = test_positive > 0

print(f"Classes with positive training examples:   {train_seen.sum().item()}/{NUM_CLASSES}")
print(f"Classes with positive validation examples: {val_seen.sum().item()}/{NUM_CLASSES}")
print(f"Classes with positive test examples:       {test_seen.sum().item()}/{NUM_CLASSES}")

print(f"\nValidation-positive classes absent from training: {(val_seen & ~train_seen).sum().item()}")
print(f"Test-positive classes absent from training:       {(test_seen & ~train_seen).sum().item()}")

print(f"\nPositive training observation/class pairs:   {int(train_positive.sum().item()):,}")
print(f"Positive validation observation/class pairs: {int(val_positive.sum().item()):,}")
print(f"Positive test observation/class pairs:       {int(test_positive.sum().item()):,}")

Classes with positive training examples:   105/304
Classes with positive validation examples: 83/304
Classes with positive test examples:       62/304

Validation-positive classes absent from training: 20
Test-positive classes absent from training:       8

Positive training observation/class pairs:   73,696
Positive validation observation/class pairs: 38,388
Positive test observation/class pairs:       12,235
